# Importing libraries and modules

In [ ]:
import numpy as np
import torch
import torch.nn as nn

In [ ]:
names = open('/content/sample_data/names').read()

In [ ]:
names_as_lst = names.splitlines()

In [ ]:
print('numero nombres:', len(names_as_lst))

numero nombres: 32033


In [ ]:
array_names_lens = np.array([len(name) for name in names_as_lst])

In [ ]:
print('máximo:', np.max(array_names_lens))
print('mean:', np.mean(array_names_lens))
print('std:', np.std(array_names_lens))

máximo: 15
mean: 6.122217712983486
std: 1.4403347813133913


# Constantes

In [ ]:
VOCAB = set(names.replace('\n', '')).union({'.', '!'})
VOCAB_SIZE = len(VOCAB)
BLOCK_SIZE = 16
EMBEDDING_DIM = 64

# Mapeos

In [ ]:
VTI = {char: i for i,char in enumerate(VOCAB)}
ITV = {VTI[char]: char for char in VTI}

# Construyendo dataset

In [ ]:
X = []
Y = []
for name in names_as_lst:
  x_name = []
  y_name = []
  new_name = '.' + name + '.' + (BLOCK_SIZE - len(name) -1) * '!'
  for idx in range(len(new_name)):
    if idx == len(new_name)-1:
      break
    x = VTI[new_name[idx]]
    y = VTI[new_name[idx+1]]
    x_name.append(x)
    y_name.append(y)
  X.append(x_name)
  Y.append(y_name)
X = torch.tensor(X) # (B,T)
Y = torch.tensor(Y) # (B,T)

In [ ]:
# 1. Definir el dispositivo (GPU si está disponible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo seleccionado: {device}")

# Mover los datasets X e Y a la GPU
X = X.to(device)
Y = Y.to(device)

Dispositivo seleccionado: cuda


### División de Datos en Train, Val (Eval) y Test
Vamos a separar de manera aleatoria el dataset completo (X, Y) con una proporción de 80% para entrenamiento, 10% para evaluación y 10% para prueba.

In [ ]:
import torch

# Fijar semilla para reproducibilidad
g = torch.Generator(device=device).manual_seed(42)
num_ejemplos = X.shape[0]

# Crear una permutación aleatoria de los índices
indices = torch.randperm(num_ejemplos, generator=g, device=device)

# Definir límites para las particiones (80% / 10% / 10%)
n1 = int(0.8 * num_ejemplos)
n2 = int(0.9 * num_ejemplos)

# Dividir índices
indices_train = indices[:n1]
indices_val = indices[n1:n2]
indices_test = indices[n2:]

# Crear subconjuntos
X_train, Y_train = X[indices_train], Y[indices_train]
X_val, Y_val = X[indices_val], Y[indices_val]
X_test, Y_test = X[indices_test], Y[indices_test]

print(f"Tamaño del dataset completo: {num_ejemplos}")
print(f"Entrenamiento (Train): {X_train.shape[0]} ejemplos")
print(f"Evaluación (Val): {X_val.shape[0]} ejemplos")
print(f"Prueba (Test): {X_test.shape[0]} ejemplos")

Tamaño del dataset completo: 32033
Entrenamiento (Train): 25626 ejemplos
Evaluación (Val): 3203 ejemplos
Prueba (Test): 3204 ejemplos


### Nuevo Ciclo de Entrenamiento con Monitoreo de Pérdida

Creamos una función rápida para evaluar las pérdidas sin calcular gradientes, y luego realizamos el entrenamiento imprimiendo las métricas cada 50 iteraciones.

In [ ]:
@torch.no_grad()
def evaluar_perdida(model, x, y):
    model.eval()
    _, loss = model(x, y)
    model.train()
    return loss.item()

# Modelo de lenguaje

In [ ]:
class BigramaNN(nn.Module):

  def __init__(
      self,
      vocab_size,
      embedding_dim
  ):
    super().__init__()
    self.embedding_layer = nn.Embedding(
        vocab_size,
        embedding_dim
    )
    self.linear_layer = nn.Linear(
        embedding_dim,
        vocab_size
    )

  def forward(self, x, target):
    embedded_x = self.embedding_layer(x) # (B, T, C)
    B, T, C = embedded_x.shape
    logits = self.linear_layer(embedded_x) # (B, T, vocab_size)
    vocab_size = logits.shape[-1]
    if target is None:
      loss = None
    else:
      loss = nn.functional.cross_entropy(logits.view(B*T,vocab_size), target.view(B*T), ignore_index=VTI['!'])
    return logits, loss

  def generate_next_step(self, idx):
    x_tensor = torch.tensor([[idx]], dtype=torch.long)
    logits, loss = self.forward(x_tensor, None) # (B=1, T=1, vocab_size)
    vector_prob = torch.softmax(logits[0, -1], dim=-1)
    idx = torch.multinomial(vector_prob, 1).item()
    return idx

In [ ]:
class BigramaNN(nn.Module):

  def __init__(
      self,
      vocab_size,
      embedding_dim
  ):
    super().__init__()
    self.embedding_layer = nn.Embedding(
        vocab_size,
        embedding_dim
    )
    self.linear_layer = nn.Linear(
        embedding_dim,
        vocab_size
    )

  def forward(self, x, target):
    embedded_x = self.embedding_layer(x) # (B, T, C)
    B, T, C = embedded_x.shape
    logits = self.linear_layer(embedded_x) # (B, T, vocab_size)
    vocab_size = logits.shape[-1]
    if target is None:
      loss = None
    else:
      loss = nn.functional.cross_entropy(logits.view(B*T,vocab_size), target.view(B*T), ignore_index=VTI['!'])
    return logits, loss

  def generate_next_step(self, idx):
    # Detectar dinámicamente el dispositivo del modelo para crear el tensor en el mismo lugar
    dev = next(self.parameters()).device
    x_tensor = torch.tensor([[idx]], dtype=torch.long, device=dev)
    logits, loss = self.forward(x_tensor, None) # (B=1, T=1, vocab_size)
    vector_prob = torch.softmax(logits[0, -1], dim=-1)
    idx = torch.multinomial(vector_prob, 1).item()
    return idx

# Declaración del modelo

In [ ]:
language_model = BigramaNN(
    VOCAB_SIZE,
    EMBEDDING_DIM
)

In [ ]:
# Instanciar el modelo y enviarlo a la GPU/CPU configurada
language_model = BigramaNN(
    VOCAB_SIZE,
    EMBEDDING_DIM
).to(device)

# Número de parámetros

In [ ]:
total_params = sum(p.numel() for p in language_model.parameters())
print(f'Número total de parámetros en el modelo: {total_params:,}')

Número total de parámetros en el modelo: 3,612


# Optimizador

In [ ]:
optimizer = torch.optim.AdamW(language_model.parameters(), lr=1e-3)

# Entrenamiento

In [ ]:
for n in range(2000):

  # Forward
  logits, loss = language_model(X, Y)

  # Set grads to None
  optimizer.zero_grad()

  # Calculate gradients
  loss.backward()

  # Update
  optimizer.step()

  print('iteration', n, 'loss', loss.item())


iteration 0 loss 2.828868865966797
iteration 1 loss 2.8215246200561523
iteration 2 loss 2.8143327236175537
iteration 3 loss 2.807286262512207
iteration 4 loss 2.8003854751586914
iteration 5 loss 2.7936267852783203
iteration 6 loss 2.7870078086853027
iteration 7 loss 2.7805230617523193
iteration 8 loss 2.7741730213165283
iteration 9 loss 2.7679531574249268
iteration 10 loss 2.761861562728882
iteration 11 loss 2.755894184112549
iteration 12 loss 2.7500500679016113
iteration 13 loss 2.7443270683288574
iteration 14 loss 2.7387197017669678
iteration 15 loss 2.7332284450531006
iteration 16 loss 2.7278497219085693
iteration 17 loss 2.7225823402404785
iteration 18 loss 2.717421054840088
iteration 19 loss 2.7123658657073975
iteration 20 loss 2.707413673400879
iteration 21 loss 2.702565908432007
iteration 22 loss 2.697815179824829
iteration 23 loss 2.693161964416504
iteration 24 loss 2.6886041164398193
iteration 25 loss 2.6841397285461426
iteration 26 loss 2.679767370223999
iteration 27 loss 2.6

In [ ]:
# Generación de nombres usando el modelo entrenado
for _ in range(1000):
  out = []
  idx = VTI['.'] # Caracter inicial
  while True:
    idx = language_model.generate_next_step(idx)
    char = ITV[idx]
    if char == '.' or char == '!':
      break
    out.append(char)
  print(''.join(out))

kimamenin
varan
riadahaerahzo
naysi
je
nginyoa
any
ien
bdremy
ailien
brsyn
h
hin
anelen
nora
s
delanaynnatre
kisiyare
thlen
richomaniayl
kiakilyzia
h
bana
ks
chi
prethruwir
teweayxfohy
th
ja
arema
s
sghemui
a
da
ka
elur
jarelosyomiesalisarys
ma
aa
an
a
rsssissemia
dainiarkanniad
ji
ma
jeateria
beiril
aist
rr
sral
aya
a
brlyoimalamichmalizan
ar
monard
nnn
dden
s
ayeamitakiorkisadelajah
shie
ilydayli
wan
ckh
nea
maje
aryorarhu
eluwafalesr
telily
z
riar
t
annlele
a
zeralyangontariogbineli
al
akh
xm
zlezulyt
chanarlene
sasennalynan
z
fasolanolol
thelitisiaste
li
koulle
anth
leyarakylailaxann
emize
i
torili
beli
dol
zraia
cahayn
akli
rone
enabrsuthoca
rirensatera
n
asmieynn
jaikalel
ko
jonamia
dambrsikesa
many
me
jhiaholukaha
sanat
ai
acyn
h
m
a
rmastyah
ar
lenan
a
eoran
karo
ante
aasesann
mxizan
hary
kaliepembetyi
kneoceelv
ecelavi
mennabe
tahl
javeinndor
sosh
ashayanaben
aynnerndril
s
saanandithannndya
rtocahrienehaleagh
provigale
se
ktth
jahmadeonoe
lueletoa
isaiondeeicch
madeph
kha
un
l

In [ ]:
def generar_nombre_aleatorio(model):
    out = []
    idx = VTI['.'] # Iniciamos con el caracter especial '.'
    while True:
        idx = model.generate_next_step(idx)
        char = ITV[idx]
        if char == '.' or char == '!': # Condición de parada
            break
        out.append(char)
    return ''.join(out)

# Ejemplo: Generar 5 nombres aleatorios con nuestro modelo
print("Nombres generados de forma aleatoria:")
for _ in range(5):
    nombre = generar_nombre_aleatorio(language_model)
    print("-", nombre)

In [ ]:
with torch.no_grad():
    logits, loss = language_model(X, Y)
    # El 'loss' calculado por CrossEntropy con reducción 'mean' es el promedio del negativo del log-likelihood
    neg_log_likelihood = loss.item()
    perplexity = np.exp(neg_log_likelihood)

print(f"Log-verosimilitud negativa promedio por carácter: {neg_log_likelihood:.4f}")
print(f"Perplejidad del modelo en el dataset: {perplexity:.4f}")

Log-verosimilitud negativa promedio por carácter: 2.4547
Perplejidad del modelo en el dataset: 11.6428


In [ ]:
import torch

# 1. Inicializar matriz de conteos para todos los caracteres del vocabulario
# Usamos las dimensiones de nuestro vocabulario
N = torch.zeros((VOCAB_SIZE, VOCAB_SIZE), dtype=torch.int32)

# 2. Llenar la matriz de conteos recorriendo el dataset X e Y
for x_row, y_row in zip(X, Y):
    for ix, iy in zip(x_row, y_row):
        # Ignoramos el token de relleno '!' para evaluar la verosimilitud real del texto
        if ix.item() == VTI['!'] or iy.item() == VTI['!']:
            continue
        N[ix.item(), iy.item()] += 1

# 3. Calcular la matriz de probabilidades empíricas (con suavizado de Laplace +1 para evitar log(0))
P = (N + 1).float()
P /= P.sum(1, keepdim=True)

# 4. Calcular la log-verosimilitud sobre todo el dataset basándonos en estas probabilidades empíricas
log_likelihood = 0.0
n_total = 0

for x_row, y_row in zip(X, Y):
    for ix, iy in zip(x_row, y_row):
        if ix.item() == VTI['!'] or iy.item() == VTI['!']:
            continue
        prob = P[ix.item(), iy.item()]
        log_prob = torch.log(prob)
        log_likelihood += log_prob.item()
        n_total += 1

neg_log_likelihood = -log_likelihood
promedio_nll = neg_log_likelihood / n_total
perplejidad_empirica = np.exp(promedio_nll)

print(f"Log-verosimilitud total (empírica): {log_likelihood:.4f}")
print(f"Log-verosimilitud negativa promedio por carácter: {promedio_nll:.4f}")
print(f"Perplejidad empírica del dataset (Bigrama directo): {perplejidad_empirica:.4f}")

Log-verosimilitud total (empírica): -560026.1796
Log-verosimilitud negativa promedio por carácter: 2.4547
Perplejidad empírica del dataset (Bigrama directo): 11.6427


In [ ]:
ITV

{0: 'd',
 1: 'l',
 2: 'v',
 3: 's',
 4: 'y',
 5: 'q',
 6: 'z',
 7: 'f',
 8: 'c',
 9: 'I',
 10: 't',
 11: 'g',
 12: 'i',
 13: 'h',
 14: 'F',
 15: 'j',
 16: 'e',
 17: 'w',
 18: 'u',
 19: 'k',
 20: 'a',
 21: 'n',
 22: 'm',
 23: 'b',
 24: 'o',
 25: 'r',
 26: 'x',
 27: 'p'}